In [39]:
import pandas as pd
import numpy as np
import faiss

from pathlib import Path

from sentence_transformers import SentenceTransformer
from llama_cpp import Llama

In [40]:
BASE_DIR = Path.cwd().parent

KNOWLEDGE_DIR = BASE_DIR / "data" / "knowledge_base"
VECTOR_DIR = BASE_DIR / "vector_store"
MODEL_DIR = BASE_DIR / "models" / "Qwen2.5-Coder-1.5B"

print("Knowledge directory:", KNOWLEDGE_DIR)
print("Vector directory:", VECTOR_DIR)
print("Model directory:", MODEL_DIR)

Knowledge directory: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /data/knowledge_base
Vector directory: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /vector_store
Model directory: /home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /models/Qwen2.5-Coder-1.5B


In [41]:
kb_df = pd.read_csv(
    KNOWLEDGE_DIR / "knowledge_base.csv"
)

print("Knowledge base shape:", kb_df.shape)

kb_df.head()

Knowledge base shape: (99, 5)


,document_id,category,title,content,text
0,KB001,Billing,Payment Failure,"If a payment fails, the customer should first ...","Payment Failure. If a payment fails, the custo..."
1,KB002,Billing,Refund Request,Customers can request a refund by contacting c...,Refund Request. Customers can request a refund...
2,KB003,Billing,Invoice Request,Customers who need an invoice should contact c...,Invoice Request. Customers who need an invoice...
3,KB013,Billing,Card Declined,"If a customer's card is declined, the customer...",Card Declined. If a customer's card is decline...
4,KB014,Billing,Insufficient Balance,A payment may fail when the selected payment a...,Insufficient Balance. A payment may fail when ...


In [42]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [43]:
knowledge_text = (
    kb_df["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

knowledge_embeddings = embedding_model.encode(
    knowledge_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

knowledge_embeddings = knowledge_embeddings.astype("float32")

print("Embedding shape:", knowledge_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (99, 384)


In [44]:
np.save(
    VECTOR_DIR / "knowledge_base_embeddings.npy",
    knowledge_embeddings
)

print("Knowledge embeddings saved.")

Knowledge embeddings saved.


In [45]:
embedding_dimension = knowledge_embeddings.shape[1]

knowledge_index = faiss.IndexFlatIP(
    embedding_dimension
)

knowledge_index.add(
    knowledge_embeddings
)

print("Index dimension:", knowledge_index.d)
print("Number of documents:", knowledge_index.ntotal)

Index dimension: 384
Number of documents: 99


In [46]:
faiss.write_index(
    knowledge_index,
    str(VECTOR_DIR / "knowledge_base.index")
)

print("Knowledge FAISS index saved.")

Knowledge FAISS index saved.


In [47]:
query = "What should I do if my payment fails?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

In [48]:
top_k = 5

similarity_scores, indices = knowledge_index.search(
    query_embedding,
    top_k
)

In [49]:
indices

array([[ 0,  4, 33,  8,  7]])

In [50]:
retrieved_documents = kb_df.iloc[
    indices[0]
].copy()

retrieved_documents.insert(
    0,
    "Rank",
    range(1, top_k + 1)
)

retrieved_documents["Similarity"] = (
    similarity_scores[0]
)

retrieved_documents[
    [
        "Rank",
        "document_id",
        "category",
        "title",
        "Similarity"
    ]
]

,Rank,document_id,category,title,Similarity
0,1,KB001,Billing,Payment Failure,0.732246
4,2,KB014,Billing,Insufficient Balance,0.620938
33,3,KB041,Refund,Refund Not Received,0.528198
8,4,KB018,Billing,Incorrect Billing Amount,0.508619
7,5,KB017,Billing,Payment Reversed,0.507041


In [51]:
def retrieve_documents(
    query,
    embedding_model,
    index,
    metadata,
    top_k=5
):
    
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    similarity_scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = metadata.iloc[
        indices[0]
    ].copy()

    results.insert(
        0,
        "Rank",
        range(1, top_k + 1)
    )

    results["Similarity"] = similarity_scores[0]

    return results

In [52]:
results = retrieve_documents(
    "How can I reset my password?",
    embedding_model,
    knowledge_index,
    kb_df,
    top_k=5
)

results[
    [
        "Rank",
        "document_id",
        "category",
        "title",
        "Similarity"
    ]
]

,Rank,document_id,category,title,Similarity
16,1,KB004,Account,Password Reset,0.676956
18,2,KB026,Account,Forgotten Username,0.433104
17,3,KB005,Account,Account Access Problem,0.404672
22,4,KB030,Account,Multiple Login Attempts,0.393963
21,5,KB029,Account,Locked Account,0.393696


In [53]:
def build_context(results):
    
    context_parts = []

    for _, row in results.iterrows():
        
        context_parts.append(
            f"Document: {row['title']}\n"
            f"Category: {row['category']}\n"
            f"Content: {row['content']}"
        )

    context = "\n\n".join(context_parts)

    return context

In [54]:
context = build_context(results)

print(context)

Document: Password Reset
Category: Account
Content: Customers who cannot access their account should use the password reset option on the account login page. They should follow the instructions sent to their registered email address. If the reset process does not work, the customer should contact customer support.

Document: Forgotten Username
Category: Account
Content: Customers who have forgotten their username should use the account recovery option if available. They may need to provide their registered email address or other account information to verify ownership.

Document: Account Access Problem
Category: Account
Content: If a customer cannot access their account, they should verify their login credentials and check whether the account email is correct. Customers should use the password recovery process when necessary. Persistent access problems should be escalated to customer support.

Document: Multiple Login Attempts
Category: Account
Content: Repeated unsuccessful login atte

In [58]:
LLM_PATH = (
    MODEL_DIR /
    "qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"
)

print(LLM_PATH)

/home/aximsoft/Downloads/Weekend_Task/AI Customer Intelligence Platform /models/Qwen2.5-Coder-1.5B/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf


In [59]:
print(LLM_PATH.exists())

True


In [60]:
llm = Llama(
    model_path=str(LLM_PATH),
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

print("Local LLM loaded.")

Local LLM loaded.


In [61]:
def create_rag_prompt(question, context):

    prompt = f"""
You are a customer support assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the context does not contain enough information to answer
the question, clearly say that the available knowledge base
does not contain enough information.

Do not invent policies, procedures, prices, dates, or other
facts that are not present in the context.

Context:
{context}

User Question:
{question}

Answer:
"""

    return prompt

In [62]:
question = "What should I do if my payment fails?"

results = retrieve_documents(
    question,
    embedding_model,
    knowledge_index,
    kb_df,
    top_k=5
)

context = build_context(results)

prompt = create_rag_prompt(
    question,
    context
)

response = llm(
    prompt,
    max_tokens=200,
    temperature=0.2,
    stop=["User Question:"]
)

answer = response["choices"][0]["text"].strip()

print(answer)

Based on the information provided in the context, if a payment fails, the customer should first verify that the payment method has sufficient balance and that the card or payment account is active. The customer can retry the payment after checking the payment details. If the payment continues to fail, the customer should contact customer support with the transaction details. The context does not provide specific instructions on how to handle other common issues like insufficient balance, refund not received, incorrect billing amount, or payment reversed. Therefore, the available knowledge base does not contain enough information to provide a specific answer to the user's question. The context only provides general guidance on what to do when a payment fails. The customer should consult the specific policies and procedures for their payment method and account to determine the most appropriate course of action. The context does not provide any information on how to handle other common is

In [63]:
def rag_answer(
    question,
    embedding_model,
    index,
    metadata,
    llm,
    top_k=5
):
    
    results = retrieve_documents(
        question,
        embedding_model,
        index,
        metadata,
        top_k=top_k
    )

    context = build_context(
        results
    )

    prompt = create_rag_prompt(
        question,
        context
    )

    response = llm(
        prompt,
        max_tokens=200,
        temperature=0.2,
        stop=["User Question:"]
    )

    answer = response[
        "choices"
    ][0]["text"].strip()

    return answer, results

In [64]:
question = "How can I reset my password?"

answer, retrieved_results = rag_answer(
    question,
    embedding_model,
    knowledge_index,
    kb_df,
    llm,
    top_k=5
)

print("QUESTION:")
print(question)

print("\nRETRIEVED DOCUMENTS:")
print(
    retrieved_results[
        [
            "Rank",
            "document_id",
            "title",
            "Similarity"
        ]
    ]
)

print("\nANSWER:")
print(answer)

QUESTION:
How can I reset my password?

RETRIEVED DOCUMENTS:
    Rank document_id                    title  Similarity
16     1       KB004           Password Reset    0.676956
18     2       KB026       Forgotten Username    0.433104
17     3       KB005   Account Access Problem    0.404672
22     4       KB030  Multiple Login Attempts    0.393963
21     5       KB029           Locked Account    0.393696

ANSWER:
To reset your password, you should follow the instructions sent to your registered email address. If the reset process does not work, you should contact customer support. If you have forgotten your username, you can use the account recovery option if available. If you have persistent access problems, you should stop repeated login attempts and use the password recovery process instead of continuing to enter uncertain credentials. If your account is locked, you should follow the account recovery instructions. If access remains blocked, support should verify the account and ass

In [65]:
test_questions = [
    "What should I do if my payment fails?",
    "How can I reset my password?",
    "What should I do if my delivery is delayed?",
    "How can I request a product replacement?",
    "How can I cancel my order?"
]

for question in test_questions:

    answer, results = rag_answer(
        question,
        embedding_model,
        knowledge_index,
        kb_df,
        llm,
        top_k=5
    )

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nTOP DOCUMENT:")
    print(results.iloc[0]["title"])

    print("\nANSWER:")
    print(answer)

QUESTION:
What should I do if my payment fails?

TOP DOCUMENT:
Payment Failure

ANSWER:
If a payment fails, the customer should first verify that the payment method has sufficient balance and that the card or payment account is active. The customer can retry the payment after checking the payment details. If the payment continues to fail, the customer should contact customer support with the transaction details. If the payment is still failing, the customer should check the available balance and verify the payment details before retrying. If the payment is still failing, the customer should contact customer support with the transaction details. If the payment is still failing, the customer should contact customer support with the transaction details. If the payment is still failing, the customer should contact customer support with the transaction details. If the payment is still failing, the customer should contact customer support with the transaction details. If the payment is still

In [66]:
question = "What is your company's international shipping tax policy?"

answer, results = rag_answer(
    question,
    embedding_model,
    knowledge_index,
    kb_df,
    llm,
    top_k=5
)

print("ANSWER:")
print(answer)

ANSWER:
The company's international shipping tax policy is not provided in the given context. The context only contains information about shipping methods, policy verification, incorrect billing amounts, and missing packages. To answer the user's question, more information about the company's international shipping tax policy is needed. The available knowledge base does not contain enough information to answer the question. The company's international shipping tax policy is not provided in the given context. The context only contains information about shipping methods, policy verification, incorrect billing amounts, and missing packages. To answer the user's question, more information about the company's international shipping tax policy is needed. The available knowledge base does not contain enough information to answer the question. The company's international shipping tax policy is not provided in the given context. The context only contains information about shipping methods, poli

In [67]:
sample_results = []

for question in test_questions:

    answer, results = rag_answer(
        question,
        embedding_model,
        knowledge_index,
        kb_df,
        llm,
        top_k=5
    )

    top_document = results.iloc[0]

    sample_results.append({
        "Question": question,
        "Top_Document": top_document["title"],
        "Similarity": top_document["Similarity"],
        "Answer": answer
    })

rag_results_df = pd.DataFrame(
    sample_results
)

rag_results_df.to_csv(
    BASE_DIR / "data" / "processed" / "phase14_rag_results.csv",
    index=False
)

rag_results_df

,Question,Top_Document,Similarity,Answer
0,What should I do if my payment fails?,Payment Failure,0.732246,"To resolve a payment failure, the customer sho..."
1,How can I reset my password?,Password Reset,0.676956,"To reset your password, you should follow the ..."
2,What should I do if my delivery is delayed?,Delayed Delivery,0.681989,"If your delivery is delayed, you should first ..."
3,How can I request a product replacement?,Product Replacement,0.678249,"To request a product replacement, customers sh..."
4,How can I cancel my order?,Order Cancellation,0.741581,"To cancel your order, you should contact custo..."
